In [ ]:
# Demo Pipeline Notebook
# Run the full AI-PQC Risk Assessment workflow step by step

import pandas as pd

# Import modules
from src.data_ingestion.ingest_logs import ingest_csv
from src.data_ingestion.schema_validator import validate_schema
from src.rule_scanner.scanner import scan_tls_config
from src.anomaly_detection.isolation_forest import train_isolation_forest, detect_anomalies
from src.risk_classification.random_forest import train_random_forest, classify_risk
from src.recommendations.recommend_engine import generate_recommendations
from src.explainability.shap_explainer import explain_model
from src.explainability.report_generator import generate_report
from src.utils.helpers import preprocess_data
from src.utils.config import FEATURE_COLUMNS, TARGET_COLUMN, REPORT_PATH

# Step 1: Ingest data
df = ingest_csv("../data/example_tls.csv")
validate_schema(df)
print("Data sample:")
display(df.head())

# Step 2: Rule-based scanning
df = scan_tls_config(df)
print("Rule findings sample:")
display(df[["protocol_version", "key_exchange_algorithm", "key_size", "rule_findings"]].head())

# Step 3: Preprocess data
df_clean = preprocess_data(df)

# Step 4: Anomaly detection
iso_model = train_isolation_forest(df_clean, features=["key_size"])
df = detect_anomalies(iso_model, df_clean, features=["key_size"])
print("Anomaly detection sample:")
display(df[["key_size", "anomaly_label"]].head())

# Step 5: Risk classification
if TARGET_COLUMN not in df.columns:
    df[TARGET_COLUMN] = df["anomaly_label"].apply(lambda x: "High" if x == -1 else "Low")

rf_model = train_random_forest(df, features=["key_size"], target=TARGET_COLUMN)
df = classify_risk(rf_model, df, features=["key_size"])
print("Risk classification sample:")
display(df[["key_size", "predicted_risk"]].head())

# Step 6: Recommendations
df = generate_recommendations(df)
print("Recommendations sample:")
display(df[["rule_findings", "recommendations"]].head())

# Step 7: Explainability
shap_values = explain_model(rf_model, df, features=["key_size"])
print("SHAP values generated.")

# Step 8: Report generation
generate_report(REPORT_PATH, findings=df["rule_findings"].tolist(), recommendations=df["recommendations"].tolist())
print(f"Report generated at {REPORT_PATH}")
